# Week 3 — Data Cleaning and EDA
Employee dataset cleaning notebook

## 1. Load and inspect the data

In [ ]:
import pandas as pd

df = pd.read_csv("data/employee_data.csv")

print(df.shape)
print(df.head(20))
print(df.info())

## 2. Clean column names

In [ ]:
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")
print(df.columns)

## 3. Check missing values

In [ ]:
missing_counts = df.isna().sum()
missing_percent = (df.isna().mean() * 100).round(2)

missing_summary = pd.DataFrame({
    "missing_count": missing_counts,
    "missing_percent": missing_percent
})
print(missing_summary)

## 4. Handle missing employee_id\nThe employee_id is essential, so rows without one are dropped.

In [ ]:
df = df.dropna(subset=["employee_id"])

## 5. Check and remove duplicates

In [ ]:
print(df.duplicated().sum())
print(df[df.duplicated()])

df = df.drop_duplicates()
print(df.shape)

## 6. Clean text values (department)
Before cleaning, look at what is actually inside the column.

In [ ]:
print(df["department"].unique())
print(df["department"].value_counts())

In [ ]:
df["department"] = df["department"].str.strip()
df["department"] = df["department"].str.upper()

df["department"] = df["department"].replace({
    "HUMAN RESOURCES": "HR",
    "INFORMATION TECHNOLOGY": "IT",
    "I.T.": "IT",
    "I T": "IT"
})

print(df["department"].value_counts())

## 7. Fix data types
`score` and `attendance` should be numbers. Some values (like "N/A" or "85%") won't convert cleanly,
so the % sign needs to be removed first.

In [ ]:
df["attendance"] = df["attendance"].astype(str).str.replace("%", "", regex=False)

df["score"] = pd.to_numeric(df["score"], errors="coerce")
df["attendance"] = pd.to_numeric(df["attendance"], errors="coerce")

print(df[["score", "attendance"]].isna().sum())

## 8. Fix salary
Salary was stored as text in some rows because of the thousands comma (e.g. "58,000").

In [ ]:
df["salary"] = df["salary"].astype(str).str.replace(",", "", regex=False)
df["salary"] = pd.to_numeric(df["salary"], errors="coerce")

print(df["salary"].isna().sum())

## 9. Detect and handle outliers
Score should be between 0 and 100. Anything outside that range is not realistic and should not be
used to calculate the median, so we treat it as missing before filling.

In [ ]:
print(df["score"].describe())

invalid_score = (df["score"] < 0) | (df["score"] > 100)
print(df[invalid_score])

df.loc[invalid_score, "score"] = pd.NA

**Common mistake:** don't delete an outlier just because it looks big. An attendance of 102%
is impossible and should be treated as missing, but a high salary might just belong to a senior employee
and could be real — always check before deciding.

In [ ]:
invalid_attendance = df["attendance"] > 100
df.loc[invalid_attendance, "attendance"] = pd.NA

## 10. Fill missing values

In [ ]:
df["score"] = df["score"].fillna(df["score"].median())
df["attendance"] = df["attendance"].fillna(df["attendance"].median())
df["salary"] = df["salary"].fillna(df["salary"].median())
df["department"] = df["department"].fillna("Unknown")

print(df.isna().sum())

## 11. Create new columns

In [ ]:
def performance_category(score):
    if score >= 90:
        return "Excellent"
    elif score >= 80:
        return "Good"
    elif score >= 60:
        return "Satisfactory"
    else:
        return "Needs Improvement"

df["performance_category"] = df["score"].apply(performance_category)
df["attendance_flag"] = df["attendance"] >= 80

## 12. Answer EDA questions

In [ ]:
print(df.groupby("department")["score"].mean())
print(df["performance_category"].value_counts())

correlation = df["score"].corr(df["attendance"])
print("Score-attendance correlation:", correlation)

## 13. Notes

- Removed 1 exact duplicate row and 1 row with a missing employee_id.
- Standardized department names: IT, HR, and Finance each had multiple spellings.
- Attendance values with a "%" sign and salary values with commas were cleaned before converting to numbers.
- Scores outside the 0-100 range and attendance above 100% were treated as invalid and replaced using the median.
- Missing scores, attendance, and salary were filled with the median of each column.